In [11]:
import numpy as np
import pandas as pd
import requests
import plotly.graph_objects as go
from astropy.time import Time
from astropy.coordinates import TEME, ITRS, EarthLocation, CartesianRepresentation
import astropy.units as u
from datetime import datetime, timezone
from sgp4.api import Satrec



# Login + Session

In [12]:
ST_BASE    = "https://www.space-track.org"
ST_LOGIN   = f"{ST_BASE}/ajaxauth/login"
ST_QUERY   = f"{ST_BASE}/basicspacedata/query"

USERNAME = "nathandarby2022@gmail.com"   # replace with your Space-Track credentials
PASSWORD = "Boobear32*Boobear32"             # consider loading from env variable

session = requests.Session()

resp = session.post(ST_LOGIN, data={"identity": "nathandarby2022@gmail.com", "password": "Boobear32*Boobear32"})

if resp.status_code == 200:
    print("Logged in to Space-Track successfully!")
else:
    print(f"Login failed: {resp.status_code}")

Logged in to Space-Track successfully!


In [13]:
resp = session.post(ST_LOGIN, data={"identity": "nathandarby2022@gmail.com", "password": "Boobear32*Boobear32"})
print(resp.status_code)

200


# SATCAT Fetch + Cache

In [14]:
import json
import os
from datetime import datetime, timezone

SATCAT_CACHE_FILE = os.path.expanduser("~/satcat_cache.json")

def get_norad_ids(intldes, session, force=False):
    """
    Get all NORAD IDs for a launch group.
    Reads from local cache if available, otherwise queries Space-Track satcat.
    """
    # Check cache first
    if os.path.exists(SATCAT_CACHE_FILE) and not force:
        with open(SATCAT_CACHE_FILE) as f:
            cache = json.load(f)
        
        if cache.get('intldes') == intldes:
            print(f"Loaded {len(cache['norad_ids'])} NORAD IDs from cache")
            print(f"   Launch:    {intldes}")
            print(f"   Cached at: {cache['fetched_at']}")
            for obj in cache['objects']:
                print(f"   {obj['NORAD_CAT_ID']} — {obj['SATNAME']}")
            return cache['norad_ids']
        else:
            print(f"Cache is for different launch ({cache.get('intldes')}) — fetching fresh")

    # Query satcat
    print(f"Querying Space-Track satcat for launch {intldes}...")
    url = (f"{ST_QUERY}/class/satcat"
           f"/INTLDES/{intldes}~~"
           f"/OBJECT_TYPE/Payload"
           f"/orderby/NORAD_CAT_ID"
           f"/format/json")
    
    resp = session.get(url)
    
    if resp.status_code != 200:
        print(f"satcat query failed: {resp.status_code}")
        return []
    
    objects = resp.json()
    norad_ids = [obj['NORAD_CAT_ID'] for obj in objects]
    
    # Save to cache
    cache = {
        'intldes':    intldes,
        'fetched_at': datetime.now(timezone.utc).isoformat(),
        'norad_ids':  norad_ids,
        'objects':    [{'NORAD_CAT_ID': obj['NORAD_CAT_ID'], 
                        'SATNAME':      obj['SATNAME']} for obj in objects]
    }
    with open(SATCAT_CACHE_FILE, 'w') as f:
        json.dump(cache, f, indent=2)
    
    print(f"Fetched and cached {len(norad_ids)} payloads for launch {intldes}")
    for obj in objects:
        print(f"   {obj['NORAD_CAT_ID']} — {obj['SATNAME']}")
    
    return norad_ids

# Current TLE Fetch + Cache

In [15]:
TLE_CACHE_FILE = os.path.expanduser("~/tle_cache.json")

def fetch_tles(norad_ids, session, chunk_size=50, force=False):
    if os.path.exists(TLE_CACHE_FILE) and not force:
        with open(TLE_CACHE_FILE) as f:
            cache = json.load(f)
        fetched_at = datetime.fromisoformat(cache['fetched_at'])
        age = datetime.now(timezone.utc) - fetched_at
        if age.total_seconds() < 3600:
            remaining = 3600 - age.total_seconds()
            print(f"Loaded TLEs from Cache")
            print(f"   Last Query:            {int(age.total_seconds()//60)} min ago")
            print(f"   Next Available Query:  {int(remaining//60)} min")
            print(f"   Total TLEs:            {len(cache['tles'])}")
            return cache['tles']
        else:
            print("Cache expired — fetching fresh TLEs")

    all_tles = []
    for i in range(0, len(norad_ids), chunk_size):
        chunk = norad_ids[i:i+chunk_size]
        ids_str = ','.join(str(n) for n in chunk)
        url = (f"{ST_QUERY}/class/gp"
               f"/NORAD_CAT_ID/{ids_str}"
               f"/decay_date/null-val"
               f"/orderby/NORAD_CAT_ID"
               f"/format/json")
        resp = session.get(url)
        print(f"   Chunk {i//chunk_size + 1}: status {resp.status_code}")
        if resp.status_code != 200 or not resp.text.strip():
            continue
        for obj in resp.json():
            if obj.get('TLE_LINE1') and obj.get('TLE_LINE2'):
                all_tles.append({
                    'name':  obj['OBJECT_NAME'],
                    'line1': obj['TLE_LINE1'],
                    'line2': obj['TLE_LINE2']
                })

    # Save to cache
    cache = {'fetched_at': datetime.now(timezone.utc).isoformat(), 'tles': all_tles}
    with open(TLE_CACHE_FILE, 'w') as f:
        json.dump(cache, f, indent=2)

    print(f"\nLoaded TLEs from Query")
    print(f"   Last Query:            just now")
    print(f"   Next Available Query:  60 min")
    print(f"   Total TLEs:            {len(all_tles)}")
    return all_tles

In [16]:
# Transporter-9 — Oct 2023
norad_ids = get_norad_ids('2023-174', session, force=False)
tles = fetch_tles(norad_ids, session, force=False)

Loaded 95 NORAD IDs from cache
   Launch:    2023-174
   Cached at: 2026-04-21T03:54:00.131847+00:00
   58256 — CONNECTA T3.1
   58257 — MANTIS
   58258 — ORBASTRO-PC-1
   58259 — LEMUR 2 BASS
   58260 — LEMUR 2 DILIGHTFUL
   58262 — PLATERO
   58263 — LEMUR 2 GOOD-VIBES
   58264 — BRO-11
   58266 — DJIBOUTI-1A
   58267 — LEMUR 2 SANITA-VETRA
   58268 — CONNECTA T3.2
   58269 — IRIS-C2
   58270 — FLOCK 4Q 1
   58271 — FLOCK 4Q 16
   58272 — TIGER-6
   58273 — FLOCK 4Q 11
   58274 — FLOCK 4Q 36
   58275 — FLOCK 4Q 12
   58276 — GENMAT-1
   58277 — TIGER-5
   58278 — FLOCK 4Q 19
   58279 — FLOCK 4Q 8
   58280 — FLOCK 4Q 28
   58281 — AETHER-1
   58282 — FLOCK 4Q 27
   58283 — FLOCK 4Q 17
   58284 — FLOCK 4Q 26
   58285 — FLOCK 4Q 31
   58286 — FLOCK 4Q 30
   58287 — PICO-01B009
   58288 — ICEYE-X31
   58291 — FALCONSAT-10
   58292 — UMBRA-08
   58293 — ICEYE-X32
   58294 — ICEYE-X34
   58295 — SPACEVAN-001
   58296 — PELICAN-1 3001
   58297 — UMBRA-07
   58298 — SPIP
   58299 — AETHER-2


# Dataframe

In [17]:
rows = []
for t in tles:
    line1 = t['line1']
    intldes_raw = line1[9:17].strip()
    year  = intldes_raw[:2]
    num   = intldes_raw[2:5]
    piece = intldes_raw[5:]
    full_year = f"19{year}" if int(year) >= 57 else f"20{year}"
    intldes = f"{full_year}-{num}{piece}"
    
    rows.append({
        'International Designator': intldes,
        'NORAD Catalog Number':     line1[2:7].strip(),
        'Name':                     t['name'],
        'TLE Line 1':               line1,
        'TLE Line 2':               t['line2']
    })

df_tles = pd.DataFrame(rows)
print(f"Total objects: {len(df_tles)}")
df_tles

Total objects: 54


,International Designator,NORAD Catalog Number,Name,TLE Line 1,TLE Line 2
0,2023-174C,58258,ORBASTRO-PC-1,1 58258U 23174C 26112.58394282 .00017790 0...,2 58258 97.3771 197.5336 0006182 24.1106 336...
1,2023-174D,58259,LEMUR 2 BASS,1 58259U 23174D 26112.14836421 .00012816 0...,2 58259 97.3839 196.1594 0006897 24.4521 335...
2,2023-174G,58262,PLATERO,1 58262U 23174G 26112.22123035 .00023665 0...,2 58262 97.3913 207.8417 0003428 312.8437 47...
3,2023-174J,58264,BRO-11,1 58264U 23174J 26112.61499275 .00004129 0...,2 58264 97.3910 189.9895 0011497 51.2882 308...
4,2023-174Q,58270,FLOCK 4Q 1,1 58270U 23174Q 26112.51185857 .00065917 0...,2 58270 97.3813 203.9529 0003805 342.2641 17...
5,2023-174R,58271,FLOCK 4Q 16,1 58271U 23174R 26112.18883909 .00016403 0...,2 58271 97.3817 199.1080 0005583 18.7679 341...
6,2023-174S,58272,TIGER-6,1 58272U 23174S 26112.51236723 .00154352 0...,2 58272 97.3863 208.4976 0003699 263.4176 96...
7,2023-174T,58273,FLOCK 4Q 11,1 58273U 23174T 26112.50777388 .00015382 0...,2 58273 97.3867 199.9669 0005210 15.6896 344...
8,2023-174U,58274,FLOCK 4Q 36,1 58274U 23174U 26112.50033179 .00023938 0...,2 58274 97.3845 203.0302 0005067 347.2203 12...
9,2023-174V,58275,FLOCK 4Q 12,1 58275U 23174V 26112.21667242 .00077342 0...,2 58275 97.3832 202.8588 0002660 307.3057 52...


# Historical TLE fetch (gp_history) (!!! Careful !!!)

In [24]:
import os
os.remove(os.path.expanduser("~/tle_history_cache.json"))
print("Cache cleared")

Cache cleared


In [25]:
# Section 5 — Historical TLE Fetch (gp_history)
# One-time query — never re-query, always load from cache

HISTORY_CACHE_FILE = os.path.expanduser("~/tle_history_cache.json")

with open(SATCAT_CACHE_FILE) as f:
    satcat_cache = json.load(f)

def fetch_historical_tles(norad_ids, start_date, end_date, session, chunk_size=50):
    """
    Fetch historical TLEs from Space-Track gp_history class.
    One-time query only — results cached permanently to disk.
    start_date, end_date: 'YYYY-MM-DD' strings
    """
    # Check cache — never re-query gp_history
    if os.path.exists(HISTORY_CACHE_FILE):
        with open(HISTORY_CACHE_FILE) as f:
            cache = json.load(f)
        print(f"Historical TLEs loaded from cache")
        print(f"   Launch:     {cache['intldes']}")
        print(f"   Epoch range: {cache['start_date']} → {cache['end_date']}")
        print(f"   Records:    {len(cache['tles'])}")
        return cache['tles']

    print(f"Querying gp_history: {start_date} → {end_date}")
    print(f"WARNING: This is a one-time query — results will be cached permanently")

    all_tles = []
    for i in range(0, len(norad_ids), chunk_size):
        chunk   = norad_ids[i:i+chunk_size]
        ids_str = ','.join(str(n) for n in chunk)

        url = (f"{ST_QUERY}/class/gp_history"
               f"/NORAD_CAT_ID/{ids_str}"
               f"/epoch/{start_date}--{end_date}"
               f"/orderby/NORAD_CAT_ID,EPOCH"
               f"/format/json")

        resp = session.get(url)
        print(f"   Chunk {i//chunk_size + 1}: status {resp.status_code} — {len(resp.json()) if resp.status_code == 200 else 0} records")

        if resp.status_code != 200 or not resp.text.strip():
            continue

        for obj in resp.json():
            if obj.get('TLE_LINE1') and obj.get('TLE_LINE2'):
                all_tles.append({
                    'name':  obj['OBJECT_NAME'],
                    'norad': obj['NORAD_CAT_ID'],
                    'epoch': obj['EPOCH'],
                    'line1': obj['TLE_LINE1'],
                    'line2': obj['TLE_LINE2']
                })

    # Save to cache permanently
    cache = {
        'intldes':    satcat_cache['intldes'],
        'start_date': start_date,
        'end_date':   end_date,
        'fetched_at': datetime.now(timezone.utc).isoformat(),
        'tles':       all_tles
    }
    with open(HISTORY_CACHE_FILE, 'w') as f:
        json.dump(cache, f, indent=2)

    print(f"\nHistorical TLEs fetched and cached permanently")
    print(f"   Total records: {len(all_tles)}")
    print(f"   Cache file:    {HISTORY_CACHE_FILE}")
    return all_tles

# Transporter-9 launched October 6, 2023
# Query first 7 days post-deployment
historical_tles = fetch_historical_tles(
    norad_ids,
    start_date='2023-11-11',
    end_date='2023-12-11',
    session=session
)

Querying gp_history: 2023-11-11 → 2023-12-11
   Chunk 1: status 200 — 1409 records
   Chunk 2: status 200 — 872 records

Historical TLEs fetched and cached permanently
   Total records: 2281
   Cache file:    /Users/nathandarby/tle_history_cache.json
